### Lakebase — Complaint (Reverse-ETL)

Creates the **`caspers_complaint`** logical Postgres database inside the shared
`${CATALOG}-caspers` Lakebase Autoscaling project (provisioned by
`stages/lakebase_project.ipynb`), then creates a synced table from
`${CATALOG}.complaints.complaint_responses` (gold Delta produced by
`stages/complaint_agent_stream.ipynb`) into Postgres.

- UC synced-table entry:   `${CATALOG}.complaints.pg_complaint_responses`
- Backing Delta storage:   `${CATALOG}._lakebase_storage.pg_complaint_responses`
- Postgres destination:    `caspers_complaint.complaints.pg_complaint_responses`

In [ ]:
%pip install --upgrade "databricks-sdk>=0.81.0"

In [ ]:
dbutils.library.restartPython()

In [ ]:
import re, sys
sys.path.append('../utils')
from uc_state import add
from lakebase_autoscale import (
    get_or_create_postgres_database,
    get_or_create_autoscale_synced_table,
    get_autoscale_synced_table,
)
import status

CATALOG = dbutils.widgets.get("CATALOG")

PROJECT_ID = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-caspers".lower())
BRANCH_PATH = f"projects/{PROJECT_ID}/branches/production"

# Lakebase Autoscale requires DNS-safe names (no underscores).
POSTGRES_DB = "caspers-complaint"
POSTGRES_SCHEMA = "complaints"

SOURCE_TABLE = f"{CATALOG}.complaints.complaint_responses"
SYNCED_TABLE = f"{CATALOG}.complaints.pg_complaint_responses"
SYNCED_BASENAME = SYNCED_TABLE.split('.')[-1]
BACKING_TABLE = f"{CATALOG}._lakebase_storage.{SYNCED_BASENAME}"

print(f"CATALOG          = {CATALOG}")
print(f"PROJECT_ID       = {PROJECT_ID}")
print(f"POSTGRES_DB      = {POSTGRES_DB}")
print(f"SOURCE_TABLE     = {SOURCE_TABLE}")
print(f"SYNCED_TABLE     = {SYNCED_TABLE}")

##### Ensure the per-component Postgres database exists

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

db, created = get_or_create_postgres_database(
    w,
    project_id=PROJECT_ID,
    database_id=POSTGRES_DB,
)
if created:
    status.ok(f"Created Postgres database {POSTGRES_DB} in {PROJECT_ID}")
else:
    status.reuse(f"Reusing Postgres database {POSTGRES_DB} in {PROJECT_ID}")

##### Create the synced table (idempotent)

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}._lakebase_storage")

existing_synced = get_autoscale_synced_table(w, SYNCED_TABLE)
if existing_synced is not None:
    status.reuse(f"Found existing synced table: {SYNCED_TABLE}")
else:
    for stale in (SYNCED_TABLE, BACKING_TABLE):
        try:
            spark.sql(f"DROP TABLE IF EXISTS {stale}")
            status.drop(f"Cleared stale Delta at {stale} (no-op if absent)")
        except Exception as e:
            status.warn(f"Could not drop {stale}: {e}")

    result = get_or_create_autoscale_synced_table(
        w,
        synced_table_name=SYNCED_TABLE,
        source_table_full_name=SOURCE_TABLE,
        primary_key_columns=["complaint_id"],
        branch=BRANCH_PATH,
        postgres_database=POSTGRES_DB,
        scheduling_policy="CONTINUOUS",
        create_database_objects_if_missing=True,
    )
    status.ok(f"Created synced table {SYNCED_TABLE}")
    add(CATALOG, "autoscale_synced_tables", {
        "synced_table_name": SYNCED_TABLE,
        "project_id": PROJECT_ID,
        "postgres_database": POSTGRES_DB,
        "postgres_schema": POSTGRES_SCHEMA,
    })

In [ ]:
status.ok("lakebase (complaint) stage complete")
print(f"   Project:           projects/{PROJECT_ID}")
print(f"   Postgres DB:       {POSTGRES_DB}")
print(f"   Synced UC entry:   {SYNCED_TABLE}")
print(f"   Postgres dest:     {POSTGRES_DB}.{POSTGRES_SCHEMA}.{SYNCED_BASENAME}")